# Project 03 — Four Linear Algebra Methods

## Notebook 01 — Data Preparation

**Blocks:** 01–05  
**Dataset:** 400 MRI images; 128 × 128 grayscale; 16,384 features per image  
**Main project directory (Google Colab):** `MyDrive/MATHEMATICS_PROJECTS/PROJECT_03_LA_ML_FOUR_METHODS/`  
**Processed files:** `data/processed/X_mri_400.npy`, `y_mri_400.npy`, `filenames_mri_400.json`

*Execution note: Run cells in order after mounting Google Drive. Notebook 01 creates the processed dataset used by the following notebooks.*


### Mathematical purpose and interpretation

The dataset is represented as a matrix $X\in\mathbb{R}^{400\times16,384}$: each row is an MRI image, and each column is a pixel-intensity feature after conversion to grayscale and resizing to $128\times128$. The selection contains 100 images from each of four source categories. Scaling intensities to $[0,1]$ makes the numerical representation consistent across the selected images.

**What this establishes:** a reproducible input representation for the later linear-algebra experiments. **What it does not establish:** representative sampling of clinical cases, image quality for diagnosis, or generalization to other MRI datasets. The original image selection and preprocessing must be held fixed when comparing methods.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS"
)

folders = [
    "data/raw_subset",
    "data/processed",
    "notebooks",
    "figures",
    "results",
    "reports"
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(
        parents=True,
        exist_ok=True
    )

print("PROJECT DIRECTORIES VERIFIED SUCCESSFULLY")
print("Location:", PROJECT_ROOT)

for folder in folders:
    print("✓", folder)


PROJECT DIRECTORIES VERIFIED SUCCESSFULLY
Location: /content/drive/MyDrive/MATHEMATICS_PROJECTS/PROJECT_03_LA_ML_FOUR_METHODS
✓ data/raw_subset
✓ data/processed
✓ notebooks
✓ figures
✓ results
✓ reports


## Block 02 — Prepare and verify the image selection


In [3]:

from pathlib import Path
import random

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS"
)

SOURCE = Path(
    "/content/drive/MyDrive/OTHER MATERIALS /"
    "PROJECT_01_BLOCK_01_04_BRAIN_TUMOR_ANN_VS_CNN/"
    "data/raw/extracted_dataset/Training"
)

DESTINATION = PROJECT_ROOT / "data/raw_subset"

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
SEED = 42
N_PER_CLASS = 100

rng = random.Random(SEED)
selected = {}

for category in CLASSES:
    source_folder = SOURCE / category

    images = sorted(
        p for p in source_folder.iterdir()
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    if len(images) < N_PER_CLASS:
        raise ValueError(
            f"Not enough images in {category}: {len(images)}"
        )

    chosen = rng.sample(images, N_PER_CLASS)
    selected[category] = chosen

    print(f"{category}: {len(chosen)} selected")

print("\nTOTAL SELECTED:",
      sum(len(v) for v in selected.values()))

print("\nOriginal files remain untouched.")
print("No images have been copied yet.")



glioma: 100 selected
meningioma: 100 selected
notumor: 100 selected
pituitary: 100 selected

TOTAL SELECTED: 400

Original files remain untouched.
No images have been copied yet.


# ***Step 3 — Copy the 400 MRI images***

In [ ]:

from pathlib import Path
import shutil
import hashlib

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS"
)

DESTINATION = PROJECT_ROOT / "data/raw_subset"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

total_copied = 0
total_verified_existing = 0

for category, images in selected.items():
    target_folder = DESTINATION / category
    target_folder.mkdir(parents=True, exist_ok=True)

    copied = 0
    verified_existing = 0

    for source_path in images:
        target_path = target_folder / source_path.name

        if target_path.exists():
            if sha256_file(source_path) != sha256_file(target_path):
                raise ValueError(
                    f"Existing file differs from source: {target_path}"
                )
            verified_existing += 1
        else:
            shutil.copy2(source_path, target_path)

            if sha256_file(source_path) != sha256_file(target_path):
                raise ValueError(
                    f"Copy verification failed: {target_path}"
                )

            copied += 1

    total_copied += copied
    total_verified_existing += verified_existing

    print(
        f"{category}: {copied} copied, "
        f"{verified_existing} existing files verified"
    )

print("\nTOTAL NEWLY COPIED:", total_copied)
print("TOTAL EXISTING VERIFIED:", total_verified_existing)
print(
    "TOTAL SELECTED ACCOUNTED FOR:",
    total_copied + total_verified_existing
)

checks_passed = (
    total_copied + total_verified_existing == 400
)

print("\nALL CELL 03 CHECKS PASSED:", checks_passed)
print("Original images were not moved or modified.")


glioma: 0 copied, 100 existing files verified
meningioma: 0 copied, 100 existing files verified
notumor: 0 copied, 100 existing files verified
pituitary: 0 copied, 100 existing files verified

TOTAL NEWLY COPIED: 0
TOTAL EXISTING VERIFIED: 400
TOTAL SELECTED ACCOUNTED FOR: 400

ALL CELL 03 CHECKS PASSED: True
Original images were not moved or modified.


## Block 03 — Dataset Integrity Verification


In [ ]:

from pathlib import Path
from PIL import Image
from collections import Counter
import hashlib

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS"
)

SOURCE = Path(
    "/content/drive/MyDrive/OTHER MATERIALS /"
    "PROJECT_01_BLOCK_01_04_BRAIN_TUMOR_ANN_VS_CNN/"
    "data/raw/extracted_dataset/Training"
)

DESTINATION = PROJECT_ROOT / "data/raw_subset"

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

counts = {}
dimensions = Counter()
errors = []
verified = 0

for category in CLASSES:
    folder = DESTINATION / category

    images = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    counts[category] = len(images)

    for image_path in images:
        original_path = SOURCE / category / image_path.name

        try:
            with Image.open(image_path) as img:
                img.load()
                dimensions[img.size] += 1

            if not original_path.is_file():
                raise FileNotFoundError("Original image missing")

            if sha256_file(image_path) != sha256_file(original_path):
                raise ValueError("SHA-256 mismatch")

            verified += 1

        except Exception as e:
            errors.append((category, image_path.name, str(e)))

print("IMAGE COUNTS")
for category, count in counts.items():
    print(f"{category}: {count}")

print("\nTOTAL IMAGES:", sum(counts.values()))
print("INTEGRITY VERIFIED:", verified)

print("\nIMAGE DIMENSIONS")
for size, count in sorted(dimensions.items()):
    print(f"{size}: {count} images")

print("\nERRORS:", len(errors))

for error in errors[:10]:
    print(error)

checks_passed = (
    all(counts[c] == 100 for c in CLASSES)
    and sum(counts.values()) == 400
    and verified == 400
    and len(errors) == 0
)

assert checks_passed, "CELL_04 verification failed."

print("\nALL BLOCK 03 CHECKS PASSED:", checks_passed)


IMAGE COUNTS
glioma: 100
meningioma: 100
notumor: 100
pituitary: 100

TOTAL IMAGES: 400
INTEGRITY VERIFIED: 400

IMAGE DIMENSIONS
(150, 198): 1 images
(173, 201): 2 images
(192, 192): 1 images
(200, 243): 1 images
(200, 252): 3 images
(201, 210): 1 images
(201, 250): 2 images
(201, 251): 7 images
(203, 243): 1 images
(205, 249): 1 images
(206, 244): 3 images
(206, 245): 1 images
(210, 213): 1 images
(214, 236): 2 images
(215, 234): 2 images
(216, 202): 1 images
(216, 293): 1 images
(218, 231): 1 images
(224, 223): 1 images
(225, 225): 20 images
(226, 212): 1 images
(227, 222): 1 images
(227, 262): 2 images
(228, 221): 3 images
(232, 217): 3 images
(233, 226): 1 images
(235, 227): 3 images
(236, 221): 1 images
(236, 236): 2 images
(236, 239): 1 images
(236, 315): 1 images
(300, 168): 1 images
(306, 365): 1 images
(312, 401): 1 images
(329, 314): 1 images
(336, 380): 1 images
(339, 340): 1 images
(355, 452): 1 images
(359, 449): 2 images
(360, 378): 1 images
(393, 400): 1 images
(414, 49

## Block 04 — Image Preprocessing and Matrix Construction


In [ ]:

from pathlib import Path
from PIL import Image
import numpy as np

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS"
)

DATA_DIR = PROJECT_ROOT / "data/raw_subset"

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
IMG_SIZE = (128, 128)
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

X_list = []
y_list = []
filenames = []

for class_index, category in enumerate(CLASSES):

    folder = DATA_DIR / category

    images = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    assert len(images) == 100, (
        f"Expected 100 images in {category}, found {len(images)}"
    )

    for image_path in images:

        with Image.open(image_path) as img:
            img = img.convert("L")
            img = img.resize(IMG_SIZE, Image.Resampling.BILINEAR)

            array = np.asarray(img, dtype=np.float32) / 255.0

        vector = array.reshape(-1)

        X_list.append(vector)
        y_list.append(class_index)
        filenames.append(f"{category}/{image_path.name}")

X = np.stack(X_list)
y = np.asarray(y_list, dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X dtype:", X.dtype)
print("Minimum pixel value:", X.min())
print("Maximum pixel value:", X.max())

print("\nClass distribution:")
for i, category in enumerate(CLASSES):
    print(category, ":", np.sum(y == i))

checks_passed = (
    X.shape == (400, 16384)
    and y.shape == (400,)
    and len(filenames) == 400
    and np.isfinite(X).all()
    and X.min() >= 0
    and X.max() <= 1
    and all(np.sum(y == i) == 100 for i in range(4))
)

assert checks_passed, "CELL_05 verification failed."

print("\nALL BLOCK 04 CHECKS PASSED:", checks_passed)


X shape: (400, 16384)
y shape: (400,)
X dtype: float32
Minimum pixel value: 0.0
Maximum pixel value: 1.0

Class distribution:
glioma : 100
meningioma : 100
notumor : 100
pituitary : 100

ALL BLOCK 04 CHECKS PASSED: True


## Block 05 — Matrix Verification and Data Persistence


In [ ]:

from pathlib import Path
import numpy as np
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS"
)

OUTPUT_DIR = PROJECT_ROOT / "data/processed"

# Basic numerical checks
n, p = X.shape

row_norms = np.linalg.norm(X, axis=1)

global_mean = float(X.mean())
global_std = float(X.std())

zero_rows = int(np.sum(row_norms == 0))
nonfinite_values = int(np.size(X) - np.isfinite(X).sum())

print("MATRIX VERIFICATION")
print("Shape:", X.shape)
print("Number of observations:", n)
print("Number of features:", p)
print("Global mean:", global_mean)
print("Global standard deviation:", global_std)
print("Zero rows:", zero_rows)
print("Non-finite values:", nonfinite_values)

print("\nMATHEMATICAL RANK BOUNDS")
print("rank(X) <=", min(n, p))
print("rank(centered X) <=", min(n - 1, p))

# Final checks
checks_passed = (
    X.shape == (400, 16384)
    and y.shape == (400,)
    and len(filenames) == 400
    and nonfinite_values == 0
    and zero_rows == 0
    and np.isfinite(global_mean)
    and np.isfinite(global_std)
)

assert checks_passed, "Block 05 verification failed."

# Save only after all checks pass
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / "X_mri_400.npy", X)
np.save(OUTPUT_DIR / "y_mri_400.npy", y)

with open(OUTPUT_DIR / "filenames_mri_400.json", "w") as f:
    json.dump(filenames, f, indent=2)

# Verify saved files
X_saved = np.load(OUTPUT_DIR / "X_mri_400.npy")
y_saved = np.load(OUTPUT_DIR / "y_mri_400.npy")

with open(OUTPUT_DIR / "filenames_mri_400.json", "r") as f:
    filenames_saved = json.load(f)

saved_checks_passed = (
    np.array_equal(X, X_saved)
    and np.array_equal(y, y_saved)
    and filenames == filenames_saved
)

assert saved_checks_passed, "Saved-file verification failed."

print("\nFILES SAVED AND VERIFIED")
print("X_mri_400.npy")
print("y_mri_400.npy")
print("filenames_mri_400.json")

print("\nALL BLOCK 05 CHECKS PASSED:", checks_passed)
print("ALL SAVED-FILE CHECKS PASSED:", saved_checks_passed)


MATRIX VERIFICATION
Shape: (400, 16384)
Number of observations: 400
Number of features: 16384
Global mean: 0.18548434972763062
Global standard deviation: 0.1962425857782364
Zero rows: 0
Non-finite values: 0

MATHEMATICAL RANK BOUNDS
rank(X) <= 400
rank(centered X) <= 399

FILES SAVED AND VERIFIED
X_mri_400.npy
y_mri_400.npy
filenames_mri_400.json

ALL BLOCK 05 CHECKS PASSED: True
ALL SAVED-FILE CHECKS PASSED: True
